In [1]:
!apt-get update -qq && apt-get install -y -qq zstd pciutils lshw ffmpeg > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q "gradio>=5.0,<6.0" chromadb pymupdf faster-whisper requests

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follow

In [2]:
import os, subprocess, time, requests

# ---------- MODEL CHOICES (used by Cell 3 too) ----------
CHAT_MODEL = "gemma3:4b"       # bigger: "gemma3:12b"
EMBED_MODEL = "embeddinggemma"
WHISPER_MODEL = "large-v3"     # lighter: "medium" or "small"
# --------------------------------------------------------

os.environ["OLLAMA_MODELS"] = "/tmp/ollama_models"
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_KEEP_ALIVE"] = "6h"

# Ollama uses GPU 0 only, so Whisper can use GPU 1 (if you have T4 x2)
ollama_env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
server = subprocess.Popen(
    ["ollama", "serve"],
    env=ollama_env,
    stdout=open("/kaggle/working/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(90):
    try:
        requests.get("http://127.0.0.1:11434")
        print("Ollama server is running")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Server did not start. Read /kaggle/working/ollama.log")

!ollama pull {CHAT_MODEL}
!ollama pull {EMBED_MODEL}
!ollama list

Ollama server is running
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling f06e91e18fc7:   2% ▕                  ▏  50 MB/2.5 GB                  pulling manifest 
pulling f06e91e18fc7:   4% ▕                  ▏  95 MB/2.5 GB                  pulling manifest 
pulling f06e91e18fc7:   7% ▕█                 ▏ 164 MB/2.5 GB                  pulling manifest 
pulling f06e91e18fc7:   9% ▕█                 ▏ 227 MB/2.5 GB                  pulling manifest 
pulling f06e91e18fc7:  11% ▕█                 ▏ 263 MB/2.5 GB                  pulling manifest 
pulling f06e91e18fc7:  13% ▕██                

In [ ]:
import os, re, json, uuid, base64, html, requests
import numpy as np
import fitz  # pymupdf
import chromadb
import gradio as gr
from faster_whisper import WhisperModel

os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

# ----------------------------- SETTINGS -----------------------------
OLLAMA = "http://127.0.0.1:11434"
VISION_MODEL = CHAT_MODEL          # reads images and scanned PDF pages (gemma3 models can see)
CHUNK_WORDS, CHUNK_OVERLAP = 150, 30
DIST_MAX = 0.65                    # Auto mode: chunks farther than this are treated as not relevant
REWRITE_QUERY = True               # helps Banglish and cross-language search
HISTORY_MESSAGES = 8               # how many earlier messages the model remembers
MAX_OCR_PAGES = 15                 # scanned PDF pages read with vision per file
CTX = 8192
BANGLISH_REPLY = "banglish"        # reply to Banglish in "banglish" or "bangla" script
AUTH = None                        # protect the public link, e.g. ("admin", "mypassword")
IMG_EXT = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
AUDIO_EXT = (".wav", ".mp3", ".m4a", ".ogg", ".oga", ".opus", ".flac", ".webm", ".aac")
# --------------------------------------------------------------------

CLIENT = chromadb.PersistentClient(path="/tmp/chroma_db")
DOCS = {}          # name -> {"type", "chunks", "head"}
_whisper = None


def get_col():
    return CLIENT.get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})


# ------------------------------- Ollama calls -------------------------------
def post(path, payload, timeout=900, stream=False):
    try:
        r = requests.post(f"{OLLAMA}{path}", json=payload, timeout=timeout, stream=stream)
    except requests.ConnectionError:
        raise RuntimeError("Ollama server is not running. Run Cell 2 again.")
    if r.status_code != 200:
        raise RuntimeError(f"Ollama error {r.status_code}: {r.text[:300]}")
    return r


def prefix(t, kind):
    if kind == "query":
        return f"task: search result | query: {t}"
    return f"title: none | text: {t}"


def embed(texts, kind):
    out = []
    for i in range(0, len(texts), 16):
        batch = [prefix(t, kind) for t in texts[i:i + 16]]
        out.extend(post("/api/embed", {"model": EMBED_MODEL, "input": batch}).json()["embeddings"])
    return out


def llm_once(messages, model=None, max_tokens=None, temperature=0.0):
    opts = {"temperature": temperature, "num_ctx": CTX}
    if max_tokens:
        opts["num_predict"] = max_tokens
    r = post("/api/chat", {"model": model or CHAT_MODEL, "messages": messages,
                           "stream": False, "options": opts, "keep_alive": "6h"})
    return r.json()["message"]["content"]


def llm_stream(messages):
    payload = {"model": CHAT_MODEL, "messages": messages, "stream": True,
               "options": {"temperature": 0.3, "num_ctx": CTX}, "keep_alive": "6h"}
    r = post("/api/chat", payload, stream=True)
    for line in r.iter_lines():
        if not line:
            continue
        d = json.loads(line)
        piece = d.get("message", {}).get("content", "")
        if piece:
            yield piece
        if d.get("done"):
            break


def vision(prompt, b64):
    return llm_once([{"role": "user", "content": prompt, "images": [b64]}], model=VISION_MODEL)


# --------------------------------- voice ------------------------------------
def get_whisper():
    global _whisper
    if _whisper is None:
        try:
            import torch
            gpu = 1 if torch.cuda.device_count() > 1 else 0
            m = WhisperModel(WHISPER_MODEL, device="cuda", device_index=gpu, compute_type="float16")
            list(m.transcribe(np.zeros(16000, dtype=np.float32))[0])   # test run
            _whisper = m
            print(f"Whisper {WHISPER_MODEL} loaded on GPU {gpu}")
        except Exception as e:
            print("Whisper GPU failed, using CPU 'small' instead:", str(e)[:200])
            _whisper = WhisperModel("small", device="cpu", compute_type="int8")
    return _whisper


def transcribe(path, lang):
    segs, _ = get_whisper().transcribe(
        path, language=None if lang == "auto" else lang, vad_filter=True, beam_size=5)
    return " ".join(s.text.strip() for s in segs).strip()


# ------------------------------- ingestion ----------------------------------
def chunk_words(text, size=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    words = text.split()
    if not words:
        return []
    out, step = [], max(1, size - overlap)
    for s in range(0, len(words), step):
        out.append(" ".join(words[s:s + size]))
        if s + size >= len(words):
            break
    return out


def add_chunks(name, kind, pieces):
    col = get_col()
    texts = [p[0] for p in pieces]
    metas = [dict(source=name, type=kind, **p[1]) for p in pieces]
    for s in range(0, len(texts), 16):
        col.add(ids=[uuid.uuid4().hex for _ in texts[s:s + 16]],
                documents=texts[s:s + 16],
                embeddings=embed(texts[s:s + 16], "doc"),
                metadatas=metas[s:s + 16])
    old = DOCS.get(name, {})
    DOCS[name] = {"type": kind, "chunks": old.get("chunks", 0) + len(texts),
                  "head": old.get("head") or texts[0]}
    return len(texts)


def ingest_pdf(path):
    name = os.path.basename(path)
    doc = fitz.open(path)
    pieces, ocr_used = [], 0
    for pno, page in enumerate(doc, 1):
        text = page.get_text().strip()
        if len(text) < 30 and ocr_used < MAX_OCR_PAGES:       # scanned page -> read with vision
            pix = page.get_pixmap(dpi=130)
            b64 = base64.b64encode(pix.tobytes("png")).decode()
            text = vision("Transcribe all the text in this image exactly as written. "
                          "Keep Bangla and English as they are. Output only the text.", b64).strip()
            ocr_used += 1
        for c in chunk_words(text):
            pieces.append((c, {"page": pno}))
    doc.close()
    if not pieces:
        raise RuntimeError("no readable text found")
    n = add_chunks(name, "pdf", pieces)
    extra = f" ({ocr_used} scanned pages read with vision)" if ocr_used else ""
    return f"{name}: {n} chunks{extra}"


def ingest_text(text, name):
    pieces = [(c, {}) for c in chunk_words(text)]
    if not pieces:
        raise RuntimeError("empty text")
    n = add_chunks(name, "text", pieces)
    return f"{name}: {n} chunks"


def ingest_image(path):
    name = os.path.basename(path)
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    desc = vision("Describe this image in detail. Then write out any visible text exactly "
                  "(Bangla or English).", b64).strip()
    pieces = [(c, {}) for c in chunk_words(f"Image file {name}. {desc}")]
    n = add_chunks(name, "image", pieces)
    return f"{name}: image read, {n} chunks"


def docs_md():
    if not DOCS:
        return "_No documents yet. Attach files with the + button, or paste text below._"
    return "\n".join(f"- 📄 **{n}** · {i['type']}, {i['chunks']} chunks" for n, i in DOCS.items())


def add_text_ui(text, name):
    text = (text or "").strip()
    if not text:
        return "Paste some text first.", docs_md(), text
    try:
        msg = ingest_text(text, (name or "pasted-text").strip())
        return "✅ " + msg, docs_md(), ""
    except Exception as e:
        return f"⚠️ {e}", docs_md(), text


def clear_all():
    try:
        CLIENT.delete_collection("docs")
    except Exception:
        pass
    DOCS.clear()
    get_col()
    return "All documents cleared.", docs_md()


# ------------------------------- retrieval ----------------------------------
DOC_WORDS = re.compile(
    r"\b(pdf|document|file|image|picture|photo|upload(?:ed)?)\b|পিডিএফ|ডকুমেন্ট|ফাইল|ছবি|আপলোড",
    re.I)


def rewrite_query(q):
    prompt = ("Rewrite the user message below as a short search query for finding information in documents. "
              "If it is Banglish (Bangla typed in English letters) or Bangla, write it in Bangla script, "
              "then ' | ', then an English translation. If it is English, just clean it up. "
              "Output ONLY the query.\n\nMessage: " + q)
    try:
        return llm_once([{"role": "user", "content": prompt}], max_tokens=80).strip().split("\n")[0]
    except Exception:
        return ""


def retrieve(question, k):
    col = get_col()
    n = col.count()
    if n == 0:
        return [], ""
    queries, rq = [question], ""
    if REWRITE_QUERY:
        rq = rewrite_query(question)
        if rq and rq.lower() != question.lower():
            queries.append(rq)
    best = {}
    for e in embed(queries, "query"):
        res = col.query(query_embeddings=[e], n_results=min(k, n))
        for i, d, m, dist in zip(res["ids"][0], res["documents"][0],
                                 res["metadatas"][0], res["distances"][0]):
            if i not in best or dist < best[i][0]:
                best[i] = (dist, d, m)
    hits = sorted(best.values(), key=lambda x: x[0])[:k]
    return hits, rq


def label_of(m):
    s = m["source"]
    if m.get("type") == "pdf":
        s += f", page {m['page']}"
    return s


# --------------------------------- chat -------------------------------------
SYSTEM_BASE = (
    "You are a smart, friendly assistant like ChatGPT. You understand and speak Bangla (বাংলা), "
    "English, and Banglish (Bangla typed in English letters). Reply in the same language as the "
    "user's latest message: Bangla script for Bangla, English for English, "
    + ("and Banglish (casual Bangla in English letters) for Banglish. " if BANGLISH_REPLY == "banglish"
       else "and Bangla script for Banglish. ")
    + "Be accurate and clear. If you do not know something, say so instead of guessing."
)
RULES_AUTO = (
    " The user may also have uploaded documents, and you may receive excerpts from them. "
    "If the excerpts help answer the question, use them and cite like [1], [2]. "
    "If they are not relevant to the question, ignore them completely and answer from your own "
    "knowledge as a normal assistant, without mentioning the documents or excerpts."
)
RULES_DOCS = (
    " Answer ONLY using the excerpts from the user's documents and cite like [1], [2]. "
    "If the answer is not in the excerpts, say clearly that you could not find it in the documents."
)
MODES = ["Auto (documents + general knowledge)", "My documents only", "General knowledge only"]


def clean_history(h):
    out = []
    for m in h[-HISTORY_MESSAGES:]:
        c = m.get("content")
        if m.get("role") in ("user", "assistant") and isinstance(c, str) and c.strip() and c != "…":
            c = re.sub(r"<details>.*?</details>", "", c, flags=re.S).strip()
            if c:
                out.append({"role": m["role"], "content": c})
    return out


def file_path(f):
    if isinstance(f, str):
        return f
    if isinstance(f, dict):
        return f.get("path") or f.get("name")
    return getattr(f, "path", None) or getattr(f, "name", None)


def sources_footer(used, rq, hits):
    items = "".join(f"<li>[{i}] {html.escape(label_of(m))} · distance {d:.2f}</li>"
                    for i, (d, t, m) in enumerate(hits, 1))
    return (f"\n\n<details><summary>Sources · {html.escape(used)}</summary>"
            f"<div style='font-size:13px;opacity:.85'>Search: {html.escape(rq or '-')}<ul>{items}</ul></div></details>")


def add_user(msg, history):
    """Step 1: show the user's message immediately and clear the input bar."""
    history = list(history or [])
    msg = msg or {}
    text = (msg.get("text") or "").strip()
    paths = [p for p in (file_path(f) for f in (msg.get("files") or [])) if p]
    lines = []
    for p in paths:
        lines.append("🎤 Voice message" if p.lower().endswith(AUDIO_EXT) else f"📎 {os.path.basename(p)}")
    display = "\n\n".join(lines + ([text] if text else []))
    if not display:
        return history, gr.MultimodalTextbox(value=None, interactive=True), None
    history.append({"role": "user", "content": display})
    return history, gr.MultimodalTextbox(value=None, interactive=False), {"text": text, "files": paths}


def bot(pending, history, mode, top_k, vlang):
    """Step 2: read files, transcribe voice, search, and stream the answer."""
    history = list(history or [])
    if not pending:
        yield history, docs_md()
        return
    text, paths = pending["text"], pending["files"]
    prior = clean_history(history[:-1])
    history.append({"role": "assistant", "content": "…"})
    yield history, docs_md()

    try:
        notes, new_docs = [], False

        # 1) read attached documents
        for p in paths:
            ext = os.path.splitext(p)[1].lower()
            if ext in AUDIO_EXT:
                continue
            base = os.path.basename(p)
            history[-1]["content"] = f"Reading **{base}** …"
            yield history, docs_md()
            try:
                if ext == ".pdf":
                    msg = ingest_pdf(p)
                elif ext in (".txt", ".md"):
                    msg = ingest_text(open(p, encoding="utf-8", errors="ignore").read(), base)
                elif ext in IMG_EXT:
                    msg = ingest_image(p)
                else:
                    msg = None
                    notes.append(f"⚠️ {base}: unsupported file type")
                if msg:
                    notes.append("✅ " + msg)
                    new_docs = True
            except Exception as e:
                notes.append(f"⚠️ {base}: {e}")

        # 2) transcribe voice
        for p in paths:
            if p.lower().endswith(AUDIO_EXT):
                history[-1]["content"] = "Listening …"
                yield history, docs_md()
                try:
                    heard = transcribe(p, vlang)
                except Exception as e:
                    heard = ""
                    notes.append(f"⚠️ voice error: {e}")
                if heard:
                    text = (text + " " + heard).strip()
                    history[-2]["content"] = history[-2]["content"].replace("🎤 Voice message", "🎤 " + heard, 1)
                else:
                    notes.append("⚠️ I could not hear any speech in that recording.")

        notes_md = "\n\n".join(notes)

        # only files were attached, no question
        if not text:
            tail = "Ask me anything about them." if new_docs else ""
            history[-1]["content"] = (notes_md + ("\n\n" if notes_md and tail else "") + tail) or "…"
            yield history, docs_md()
            return

        # 3) decide what to use
        docs_only = mode == MODES[1]
        general = mode == MODES[2]
        has_docs = get_col().count() > 0

        if docs_only and not has_docs:
            history[-1]["content"] = "There are no documents yet. Attach a PDF, image, or text with the + button first."
            yield history, docs_md()
            return

        history[-1]["content"] = (notes_md + "\n\n" if notes_md else "") + "Thinking …"
        yield history, docs_md()

        hits, rq, used = [], "", "General knowledge"
        if has_docs and not general:
            hits, rq = retrieve(text, int(top_k))
            directed = new_docs or bool(DOC_WORDS.search(text))
            if directed:        # "what is the pdf about" -> also add the start of every document
                seen = {h[1] for h in hits}
                for name, info in DOCS.items():
                    if info["head"] not in seen:
                        hits.append((0.0, info["head"], {"source": name, "type": info["type"], "page": 1}))
            elif not docs_only:
                hits = [h for h in hits if h[0] <= DIST_MAX]

        if hits:
            ctx = "\n\n".join(f"[{i}] ({label_of(m)})\n{t}" for i, (d, t, m) in enumerate(hits, 1))
            user_content = f"Excerpts from the user's documents:\n{ctx}\n\nUser message: {text}"
            system = SYSTEM_BASE + (RULES_DOCS if docs_only else RULES_AUTO)
            used = "Documents only" if docs_only else "Documents + general knowledge"
        else:
            user_content, system = text, SYSTEM_BASE
            if docs_only:
                system += RULES_DOCS
                user_content = "Excerpts from the user's documents: (none found)\n\nUser message: " + text

        # 4) stream the answer
        msgs = [{"role": "system", "content": system}] + prior + [{"role": "user", "content": user_content}]
        head = (notes_md + "\n\n") if notes_md else ""
        answer = ""
        for tok in llm_stream(msgs):
            answer += tok
            history[-1]["content"] = head + answer
            yield history, docs_md()
        footer = sources_footer(used, rq, hits) if hits else ""
        history[-1]["content"] = head + answer + footer
        yield history, docs_md()

    except Exception as e:
        history[-1]["content"] = f"⚠️ Error: {e}"
        yield history, docs_md()


# ---------------------------------- UI --------------------------------------
CSS = """
footer {display: none !important;}
.gradio-container {max-width: 100% !important; padding: 0 !important;}
#sidebar {background: var(--background-fill-secondary); border-right: 1px solid var(--border-color-primary);
          padding: 16px !important; min-height: 100vh;}
#main {max-width: 880px; margin: 0 auto; padding: 8px 16px !important;}
#chatbot {border: none !important; box-shadow: none !important; background: transparent !important;}
#chatbot .message.user {background: var(--background-fill-secondary) !important; border: none !important;
                        border-radius: 22px !important;}
#chatbot .message.bot {background: transparent !important; border: none !important; box-shadow: none !important;}
#chatbot .message {font-size: 16px; line-height: 1.65;}
#chat_input {border-radius: 28px !important; box-shadow: 0 2px 14px rgba(0,0,0,.10);}
#new_chat {border-radius: 12px !important;}
details summary {cursor: pointer; opacity: .8; font-size: 13px;}
"""

theme = gr.themes.Base(
    primary_hue="emerald", neutral_hue="gray",
    font=[gr.themes.GoogleFont("Inter"), gr.themes.GoogleFont("Noto Sans Bengali"),
          "ui-sans-serif", "system-ui", "sans-serif"],
).set(
    button_primary_background_fill="#10a37f",
    button_primary_background_fill_hover="#0e8f6f",
    button_primary_text_color="white",
)

WELCOME = ("## How can I help you today?\n\n"
           "Ask in **বাংলা**, English, or Banglish.\n\n"
           "➕ Attach PDFs, images, or text &nbsp;·&nbsp; 🎤 Record your voice &nbsp;·&nbsp; "
           "Or just chat like normal")

with gr.Blocks(theme=theme, css=CSS, title="Bangla / English / Banglish Assistant") as demo:
    pending = gr.State(None)
    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=260, elem_id="sidebar"):
            gr.Markdown("### 💬 Assistant")
            btn_new = gr.Button("＋  New chat", elem_id="new_chat")
            gr.Markdown("**Your documents**")
            docs_box = gr.Markdown(docs_md())
            with gr.Accordion("Paste long text", open=False):
                name_box = gr.Textbox(label="Name", value="pasted-text")
                text_box = gr.Textbox(label="Text", lines=6)
                btn_text = gr.Button("Add text")
                paste_status = gr.Markdown()
            with gr.Accordion("Settings", open=False):
                mode = gr.Radio(MODES, value=MODES[0], label="Answer from")
                vlang = gr.Radio(["auto", "bn", "en"], value="auto", label="Voice language")
                topk = gr.Slider(2, 10, value=5, step=1, label="Chunks to retrieve")
            btn_clear = gr.Button("Clear documents", variant="stop")

        with gr.Column(scale=4, elem_id="main"):
            chatbot = gr.Chatbot(type="messages", show_label=False, elem_id="chatbot",
                                 height="calc(100vh - 190px)", placeholder=WELCOME,
                                 show_copy_button=True)
            chat_input = gr.MultimodalTextbox(
                elem_id="chat_input", show_label=False, interactive=True,
                placeholder="Message… (বাংলা / English / Banglish)",
                sources=["upload", "microphone"], file_count="multiple",
                file_types=[".pdf", ".txt", ".md", "image", "audio"],
                lines=1, max_lines=6)

    submit = chat_input.submit(add_user, [chat_input, chatbot], [chatbot, chat_input, pending])
    submit.then(bot, [pending, chatbot, mode, topk, vlang], [chatbot, docs_box]) \
          .then(lambda: gr.MultimodalTextbox(interactive=True), None, [chat_input])

    btn_new.click(lambda: [], None, [chatbot])
    btn_text.click(add_text_ui, [text_box, name_box], [paste_status, docs_box, text_box])
    btn_clear.click(clear_all, None, [paste_status, docs_box])
    demo.load(docs_md, None, docs_box)

# ------------------------------- start it ------------------------------------
print("Loading Whisper ...")
get_whisper()
try:
    post("/api/generate", {"model": CHAT_MODEL, "keep_alive": "6h"})   # warm up the chat model
except Exception as e:
    print("Warm-up skipped:", e)

demo.queue()
demo.launch(share=True, debug=True, auth=AUTH, show_api=False)

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
/tmp/ipykernel_49/1463074848.py:463: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CSS, title="Bangla / English / Banglish Assistant") as demo:
/tmp/ipykernel_49/1463074848.py:463: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CSS, title="Bangla / English / Banglish Assistant") as demo:
/tmp/ipykernel_49/1463074848.py:483: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(type="messages", show_label=False, elem_id="chatbo

Loading Whisper ...


Whisper large-v3 loaded on GPU 1


/tmp/ipykernel_49/1463074848.py:511: DeprecationWarning: The 'show_api' parameter in launch() will be removed in Gradio 6.0. You will need to use the 'footer_links' parameter instead. To replicate show_api=False, In Gradio 6.0, use footer_links=['gradio', 'settings'].
  demo.launch(share=True, debug=True, auth=AUTH, show_api=False)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://a5705da687b8119095.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# demo.close()
# server.terminate()